# AI-Lake + Apache Flink

The ailake Flink connector (`io.ailake.flink`, `ailake-flink/`) exposes AI-Lake tables as Flink SQL `CREATE TABLE ... WITH ('connector'='ailake', ...)` sources and sinks — vector search, `search.mode=full` (search + full row, no `JOIN`), FTS/hybrid text search, and INSERT (write), all backed directly by `libailake_jni.so`.

**Prerequisite:** start the flink profile:
```bash
docker compose -f tests/docker/compose-demo.yml --profile flink up -d
```
Flink Web UI: **http://localhost:8082**

**Why this notebook shells out to `sql-client.sh` instead of using a Python driver:** the query vector/text is passed to the connector as a Flink **job parameter** (`-Dpipeline.global-job-parameters.ailake.query.vector=...`), a process-launch-time flag, not a SQL-level construct — there's no `SET SESSION` equivalent for job parameters in Flink (unlike Trino), and setting it via SQL's `SET 'pipeline.global-job-parameters' = 'key:value'` breaks the moment the value itself contains a comma (which every query vector does). PyFlink was evaluated and rejected — `apache-flink==1.18.1` pins `numpy==1.21.4` via its `apache-beam` dependency, which requires Python <3.11 and cannot install on this image's Python 3.12. The Dockerfile instead installs a plain Flink **client** distribution (`FLINK_CLIENT_HOME=/opt/flink-client`, `execution.target: remote`) that submits to the separate `flink` compose service over the network — no local minicluster, no Docker-socket access needed from Jupyter.

**What this notebook covers:**
1. Run SQL against the remote Flink cluster via `sql-client.sh`
2. `search` table — pointer-only vector search (`ailake_search_json`)
3. `search.mode='full'` — search + full row, no `JOIN` (`ailake_scan_json`, Fase 11)
4. FTS / hybrid search via the `ailake.query.text` job parameter
5. Write — `INSERT INTO` an `ailake` sink table
6. Two real bugs found and fixed getting this far (§ Key takeaway)

## 0. Setup — `run_flink_sql()` helper

In [ ]:
import json
import os
import pathlib
import subprocess
import tempfile

FLINK_HOST = os.environ.get('FLINK_HOST', 'flink')
FLINK_PORT = os.environ.get('FLINK_PORT', '8081')
FLINK_CLIENT = os.environ.get('FLINK_CLIENT_HOME', '/opt/flink-client') + '/bin/sql-client.sh'
TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')

with open(str(pathlib.Path(TABLE_PATH).parent / 'demo_query.json')) as f:
    demo_q = json.load(f)
QUERY_VEC_CSV = ','.join(str(x) for x in demo_q['query_vector'])


def run_flink_sql(sql: str, job_params: dict | None = None, timeout: int = 60) -> str:
    """Write `sql` to a temp file and run it through the remote Flink SQL Client.

    `job_params` become `-Dpipeline.global-job-parameters.<key>=<value>` flags — the
    only way to set `ailake.query.vector`/`ailake.query.text`/etc (see notebook intro).
    Returns combined stdout+stderr (the SQL Client prints results as ASCII tables).
    """
    with tempfile.NamedTemporaryFile('w', suffix='.sql', delete=False) as f:
        f.write("SET 'sql-client.execution.result-mode' = 'TABLEAU';\n")
        f.write(sql)
        script_path = f.name
    cmd = [
        FLINK_CLIENT,
        f'-Drest.address={FLINK_HOST}',
        f'-Drest.port={FLINK_PORT}',
    ]
    for k, v in (job_params or {}).items():
        cmd.append(f'-Dpipeline.global-job-parameters.{k}={v}')
    cmd += ['-f', script_path]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    os.unlink(script_path)
    return result.stdout + result.stderr


try:
    import socket
    with socket.create_connection((FLINK_HOST, int(FLINK_PORT)), timeout=5):
        print(f'OK: Flink reachable at {FLINK_HOST}:{FLINK_PORT}')
    FLINK_UP = True
except OSError as exc:
    print(f'Flink not reachable at {FLINK_HOST}:{FLINK_PORT}: {exc}')
    print('Start it: docker compose -f tests/docker/compose-demo.yml --profile flink up -d')
    FLINK_UP = False

## 1. `search` table — pointer-only vector search

`ailake_search_json` — same `(row_id, distance, file_path)` shape as `ailake.search()` in `01_ailake_demo.ipynb`. Query vector via the `ailake.query.vector` job parameter.

In [ ]:
if not FLINK_UP:
    print('SKIP')
else:
    sql = f"""
CREATE TABLE demo_search (
  row_id    BIGINT,
  distance  FLOAT,
  file_path STRING
) WITH (
  'connector'     = 'ailake',
  'warehouse'     = '{TABLE_PATH}',
  'namespace'     = 'default',
  'table-name'    = 'table',
  'vector.column' = 'embedding',
  'vector.dim'    = '{demo_q["dim"]}',
  'search.top-k'  = '5'
);
SELECT * FROM demo_search;
"""
    print(run_flink_sql(sql, job_params={'ailake.query.vector': QUERY_VEC_CSV}))

## 2. `search.mode='full'` — search + full row, no `JOIN`

`ailake_scan_json` (Fase 11) — columns come straight from the `CREATE TABLE` declaration (looked up by name in the scan response); the last declared column must be `_distance`. Closes the same "pointer-only results need a manual `JOIN` to get real columns back" gap as DuckDB's `ailake_scan()` and Spark's `AilakeNative.scan()`.

In [ ]:
if not FLINK_UP:
    print('SKIP')
else:
    sql = f"""
CREATE TABLE demo_search_full (
  text      STRING,
  _distance FLOAT
) WITH (
  'connector'      = 'ailake',
  'warehouse'      = '{TABLE_PATH}',
  'namespace'      = 'default',
  'table-name'     = 'table',
  'vector.column'  = 'embedding',
  'vector.dim'     = '{demo_q["dim"]}',
  'search.mode'    = 'full',
  'search.top-k'   = '5'
);
SELECT * FROM demo_search_full;
"""
    print(run_flink_sql(sql, job_params={'ailake.query.vector': QUERY_VEC_CSV}))

## 3. FTS / hybrid search — `ailake.query.text` job parameter

`query.text` alone → pure BM25/Tantivy full-text search. `query.vector` + `query.text` together → hybrid BM25+vector RRF fusion (`ailake.hybrid.weight`, default 0.5). Uses the `fts` fixture table (`init_demo.py`, `fts_text_columns=["text"]`).

In [ ]:
if not FLINK_UP:
    print('SKIP')
else:
    fts_path = str(pathlib.Path(TABLE_PATH).parent / 'ailake_fts')
    sql = f"""
CREATE TABLE demo_fts_search (
  row_id    BIGINT,
  distance  FLOAT,
  file_path STRING
) WITH (
  'connector'     = 'ailake',
  'warehouse'     = '{fts_path}',
  'namespace'     = 'default',
  'table-name'    = 'table',
  'vector.column' = 'embedding',
  'vector.dim'    = '{demo_q["dim"]}',
  'search.top-k'  = '5'
);
SELECT * FROM demo_fts_search;
"""
    print(run_flink_sql(sql, job_params={'ailake.query.text': 'machine learning'}))

## 4. Write — `INSERT INTO` an `ailake` sink table

The `docs_ingest` DDL shape (`id BIGINT, embedding ARRAY<FLOAT>, ...`) is unrelated to the search DDL shapes above — same physical table, two different `CREATE TABLE` statements, exactly like Spark/Trino's separate `ingest`/`search` tables. `INSERT` submits a real (short-lived, batch-mode) Flink job — the SQL Client returns a job ID immediately and the statement itself does not block for completion, so this cell polls the REST API for the job to finish before reading back.

In [ ]:
if not FLINK_UP:
    print('SKIP')
else:
    import shutil
    import time
    import urllib.request

    WRITE_PATH = str(pathlib.Path(TABLE_PATH).parent / 'nb_flink_write')
    shutil.rmtree(WRITE_PATH, ignore_errors=True)

    write_sql = f"""
SET 'execution.runtime-mode' = 'batch';
CREATE TABLE demo_ingest (
  id         BIGINT,
  embedding  ARRAY<FLOAT>,
  chunk_text STRING
) WITH (
  'connector'     = 'ailake',
  'warehouse'     = '{WRITE_PATH}',
  'namespace'     = 'default',
  'table-name'    = 'table',
  'vector.column' = 'embedding',
  'vector.dim'    = '4',
  'vector.metric' = 'cosine'
);
INSERT INTO demo_ingest VALUES
  (0, ARRAY[0.1, 0.2, 0.3, 0.4], 'row zero'),
  (1, ARRAY[0.5, 0.6, 0.7, 0.8], 'row one');
"""
    print(run_flink_sql(write_sql))

    # Poll the REST API for the INSERT job to finish (a few seconds, single-node cluster).
    for _ in range(20):
        with urllib.request.urlopen(f'http://{FLINK_HOST}:{FLINK_PORT}/jobs/overview', timeout=5) as resp:
            jobs = json.load(resp)['jobs']
        insert_jobs = [j for j in jobs if 'demo_ingest' in j['name']]
        if insert_jobs and insert_jobs[-1]['state'] == 'FINISHED':
            print(f"INSERT job finished: {insert_jobs[-1]['name']}")
            break
        time.sleep(1)

    read_sql = f"""
CREATE TABLE demo_write_check (
  row_id    BIGINT,
  distance  FLOAT,
  file_path STRING
) WITH (
  'connector'     = 'ailake',
  'warehouse'     = '{WRITE_PATH}',
  'namespace'     = 'default',
  'table-name'    = 'table',
  'vector.column' = 'embedding',
  'vector.dim'    = '4',
  'search.top-k'  = '5'
);
SELECT * FROM demo_write_check;
"""
    print(run_flink_sql(read_sql, job_params={'ailake.query.vector': '0.1,0.2,0.3,0.4'}))

## Key takeaway

Every mode above — `search`, `search.mode='full'`, FTS/hybrid, and `INSERT` — is a real, executed call against `libailake_jni.so` through the ailake Flink connector on a live Flink 1.18 cluster. Of the three JVM plugins (Spark/Trino/Flink) exercised across this demo, Flink and Trino both work end-to-end (Spark: 7/9 methods, 2 undemonstrable via raw py4j reflection; Trino: planning *and* `SELECT` execution both work — two Jackson serialization bugs in Trino's internal task codec found and fixed, see `04_trino.ipynb` §13).

Two real bugs were found and fixed getting here, neither previously caught by any test in this repo (no test exercised a real multi-process Flink cluster before this):

- **`search.mode='full'` failed every query with `NotSerializableException: org.apache.flink.table.catalog.ResolvedSchema`** — `AilakeScanInputFormat` held the full `ResolvedSchema` object as a field, and Flink serializes `InputFormat` instances to ship them to TaskManagers (even in single-container standalone mode — the JobManager and TaskManager still communicate over the same internal RPC/serialization path). Fixed by extracting only the serializable bits needed (`List<ScanColumnSpec>` — name + `LogicalTypeRoot`, both actually `Serializable`) at planning time instead of holding the schema object itself.
- **`search.mode='full'` silently dropped the first (lowest-distance) result row** — `search.top-k=1` returned 0 rows, `top-k=5` returned 4. Traced to a manual `position: Int` field driving `reachedEnd()`/`nextRecord()`, which loses the first element somewhere in Flink's `InputFormatSourceFunction` driver loop (not root-caused further — Flink-internal, undocumented). The sibling `search`-mode `InputFormat` uses a plain `Iterator<T>` instead and never showed this bug; switching `AilakeScanInputFormat` to the same iterator-based pattern (materializing all rows in `open()`, matching the already-correct sibling implementation) fixed it — confirmed with `top-k` values 1 through 5, cross-checked against `ailake_scan_json` called directly (always correct) to isolate the bug to the Flink layer specifically.

See `CHANGELOG.md` for the full write-up of both fixes.